**Name**: **Anant Chauhan**

**B.Tech (CSE)**

In [1]:
from google.colab import files
uplaoded=files.upload()

Saving Daily_Cal_Req.csv to Daily_Cal_Req.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [3]:
df = pd.read_csv('Daily_Cal_Req.csv')

In [4]:
df.head()


,Age,Gender,Height_cm,Weight_kg,Activity_Level,Sleep_Hours,Water_Intake_L,Goal,Diet_Type,Daily_Calorie_Requirement
0,58.0,Male,151.0,80.0,Light,5.6,3.6,Gain,Veg,2306
1,46.0,Female,150.0,65.0,Active,6.2,1.5,Maintain,Veg,2064
2,53.0,Female,190.0,91.0,Very Active,5.5,1.2,Gain,Non-Veg,3476
3,60.0,Female,154.0,66.0,Very Active,8.1,1.6,Gain,Vegan,2207
4,35.0,Male,163.0,85.0,Light,7.8,2.4,Gain,Jain,2636


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Age                        225 non-null    float64
 1   Gender                     225 non-null    object 
 2   Height_cm                  250 non-null    float64
 3   Weight_kg                  250 non-null    float64
 4   Activity_Level             250 non-null    object 
 5   Sleep_Hours                250 non-null    float64
 6   Water_Intake_L             250 non-null    float64
 7   Goal                       250 non-null    object 
 8   Diet_Type                  250 non-null    object 
 9   Daily_Calorie_Requirement  250 non-null    int64  
dtypes: float64(5), int64(1), object(4)
memory usage: 19.7+ KB


In [6]:
df.describe()

,Age,Height_cm,Weight_kg,Sleep_Hours,Water_Intake_L,Daily_Calorie_Requirement
count,225.000000,250.00000,250.000000,250.000000,250.000000,250.00000
mean,40.066667,170.50800,78.168000,7.158000,2.775200,2462.46000
std,12.006695,11.44438,18.307606,1.424272,0.965564,553.84209
min,18.000000,150.00000,45.000000,4.500000,1.000000,1167.00000
25%,30.000000,161.00000,64.000000,6.000000,2.000000,2088.50000
50%,40.000000,171.00000,81.000000,7.200000,2.700000,2404.00000
75%,51.000000,180.00000,91.000000,8.475000,3.600000,2823.50000
max,60.000000,190.00000,110.000000,9.500000,4.500000,4107.00000


**1A. Most Preferred Diet Type**

In [7]:
diet_counts = df['Diet_Type'].value_counts()

fig = px.bar(
    x=diet_counts.index,
    y=diet_counts.values,
    color=diet_counts.index,
    title='Distribution of Diet Types',
    labels={'x': 'Diet Type', 'y': 'Count'},
    text=diet_counts.values
)
fig.update_traces(textposition='outside')
fig.show()

print("Most Preferred Diet:", diet_counts.idxmax())
print(diet_counts)

Most Preferred Diet: Veg
Diet_Type
Veg        87
Vegan      61
Non-Veg    51
Jain       51
Name: count, dtype: int64


**1B. Most Common Physical Activity Level**

In [8]:
activity_counts = df['Activity_Level'].value_counts()

fig = px.pie(
    values=activity_counts.values,
    names=activity_counts.index,
    title='Activity Level Distribution',
    hole=0.4
)
fig.show()

print("Most Common Activity Level:", activity_counts.idxmax())

Most Common Activity Level: Light


**1C. Predominant Age Group**

In [9]:
df['Age_Group'] = pd.cut(df['Age'], bins=[0,18,25,35,45,60,100],
                          labels=['<18','18-25','26-35','36-45','46-60','60+'])

age_group_counts = df['Age_Group'].value_counts().sort_index()

fig = px.bar(
    x=age_group_counts.index.astype(str),
    y=age_group_counts.values,
    color=age_group_counts.index.astype(str),
    title='Age Group Distribution',
    labels={'x': 'Age Group', 'y': 'Count'},
    text=age_group_counts.values
)
fig.update_traces(textposition='outside')
fig.show()

print("Predominant Age Group:", age_group_counts.idxmax())

Predominant Age Group: 46-60


**1D. Male vs Female Ratio**

In [10]:
gender_counts = df['Gender'].value_counts()
fig = px.pie(
    values=gender_counts.values,
    names=gender_counts.index,
    title='Gender Distribution',
    color_discrete_map={'Male': '#636EFA', 'Female': '#EF553B'}
)
fig.show()
ratio = gender_counts['Male'] / gender_counts['Female']
print(f"Male:Female Ratio = {ratio:.2f}:1")
print(gender_counts)

Male:Female Ratio = 1.39:1
Gender
Male      131
Female     94
Name: count, dtype: int64


**2A. Which Goal Requires Highest Calorie Intake?**

In [11]:
goal_calories = df.groupby('Goal')['Daily_Calorie_Requirement'].mean().sort_values(ascending=False).reset_index()

fig = px.bar(
    goal_calories,
    x='Goal',
    y='Daily_Calorie_Requirement',
    color='Goal',
    title='Average Daily Calories by Goal',
    text=goal_calories['Daily_Calorie_Requirement'].round(0)
)
fig.update_traces(textposition='outside')
fig.show()

print(goal_calories)
print("Highest Calorie Goal:", goal_calories.iloc[0]['Goal'])

       Goal  Daily_Calorie_Requirement
0      Gain                2615.526786
1  Maintain                2436.463768
2      Lose                2240.000000
Highest Calorie Goal: Gain


**2B. Is Calorie Requirement Affected by Age?**

In [12]:
fig = px.scatter(
    df,
    x='Age',
    y='Daily_Calorie_Requirement',
    trendline='ols',  # regression line dikhayega
    color='Gender',
    title='Age vs Daily Calorie Requirement',
    opacity=0.6
)
fig.show()

# Correlation check
corr = df['Age'].corr(df['Daily_Calorie_Requirement'])
print(f"Pearson Correlation (Age vs Calories): {corr:.3f}")

Pearson Correlation (Age vs Calories): -0.113


**2C. Water Intake by Activity Level**

In [13]:
px.box(
    df,
    x='Activity_Level',
    y='Water_Intake_L',
    color='Activity_Level',
    title='Water Intake by Activity Level'
).show()
water_activity = df.groupby('Activity_Level')['Water_Intake_L'].mean().reset_index()
print(water_activity)

  Activity_Level  Water_Intake_L
0         Active        2.959091
1          Light        2.701515
2       Moderate        2.660000
3      Sedentary        2.957778
4    Very Active        2.650000


**3A. Calories by Activity Level and Gender**

In [14]:
fig = px.box(
    df,
    x='Activity_Level',
    y='Daily_Calorie_Requirement',
    color='Gender',
    title='Calories by Activity Level and Gender',
    points='outliers'
)
fig.show()

# OR grouped bar
calories_grouped = df.groupby(['Activity_Level', 'Gender'])['Daily_Calorie_Requirement'].mean().reset_index()
fig2 = px.bar(
    calories_grouped,
    x='Activity_Level',
    y='Daily_Calorie_Requirement',
    color='Gender',
    barmode='group',
    title='Avg Calories by Activity Level & Gender'
)
fig2.show()

**3B. Correlation Heatmap**

In [15]:
# Encode categorical columns first
df_encoded = df.copy()
le = LabelEncoder()
cat_cols = ['Gender', 'Activity_Level', 'Diet_Type', 'Goal']
for col in cat_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])

# Drop non-numeric
numeric_df = df_encoded.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

fig = px.imshow(
    corr_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    title='Correlation Heatmap',
    height=600,
    width=800,
    aspect='auto'
)
fig.show()

**3C. Treemap — Goal, Activity_Level, Diet_Type**

In [16]:
treemap_data = df.groupby(['Goal', 'Activity_Level', 'Diet_Type']).size().reset_index(name='Count')

fig = px.treemap(
    treemap_data,
    path=['Goal', 'Activity_Level', 'Diet_Type'],
    values='Count',
    color='Count',
    color_continuous_scale='Viridis',
    height=600,
    width=800,
    title='Treemap: Goal → Activity Level → Diet Type'
)
fig.update_layout(margin=dict(t=50, l=25, r=25, b=25))
fig.show()

**3D. Sunburst Chart — Goal, Activity_Level, Diet_Type**

In [17]:
sunburst_data = df.groupby(['Goal', 'Activity_Level', 'Diet_Type']).size().reset_index(name='Count')

fig = px.sunburst(
    sunburst_data,
    path=['Goal', 'Activity_Level', 'Diet_Type'],
    values='Count',
    color='Count',
    color_continuous_scale='RdYlGn',
    height=600,
    width=800,
    title='Sunburst: Goal → Activity Level → Diet Type'
)
fig.show()

**QUESTION 4 — Linear Regression (Multiple Variables)**

In [18]:
df_model = df.copy()
le = LabelEncoder()
cat_cols = ['Gender', 'Activity_Level', 'Diet_Type', 'Goal']
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])
if 'Age_Group' in df_model.columns:
    df_model.drop('Age_Group', axis=1, inplace=True)

X = df_model.drop('Daily_Calorie_Requirement', axis=1)
y = df_model['Daily_Calorie_Requirement']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
from sklearn.impute import SimpleImputer

model = LinearRegression()

# Impute missing values before fitting
imputer = SimpleImputer(strategy='mean') # Using mean imputation

# Fit imputer on training data and transform both train and test
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Fit the model with imputed data
model.fit(X_train_imputed, y_train)

# Predict with imputed data
y_pred = model.predict(X_test_imputed)

In [20]:
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("="*40)
print(f"  MSE  : {mse:.2f}")
print(f"  RMSE : {rmse:.2f}")
print(f"  MAE  : {mae:.2f}")
print(f"  R²   : {r2:.4f}")
print("="*40)

  MSE  : 301673.30
  RMSE : 549.25
  MAE  : 446.25
  R²   : -0.0384


**Interactive Visualization — Actual vs Predicted**

In [21]:
results_df = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred, 'Index': range(len(y_test))})

# Plot 1: Scatter — Actual vs Predicted
fig1 = px.scatter(
    results_df,
    x='Actual',
    y='Predicted',
    title='Actual vs Predicted Calories',
    opacity=0.7,
    height=600,
    width=800,
    color_discrete_sequence=['#00CC96']
)
fig1.add_shape(type='line', x0=y_test.min(), y0=y_test.min(),
               x1=y_test.max(), y1=y_test.max(),
               line=dict(color='red', dash='dash'))
fig1.show()

# Plot 2: Line Chart — Comparison
fig2 = go.Figure()
fig2.add_trace(go.Scatter(y=y_test.values[:50], mode='lines+markers', name='Actual', line=dict(color='blue')))
fig2.add_trace(go.Scatter(y=y_pred[:50], mode='lines+markers', name='Predicted', line=dict(color='orange', dash='dot')))
fig2.update_layout(title='Actual vs Predicted (First 50 Samples)', xaxis_title='Sample Index', yaxis_title='Calories')
fig2.show()

# Plot 3: Residuals
residuals = y_test.values - y_pred
fig3 = px.histogram(residuals, nbins=30, title='Residual Distribution', color_discrete_sequence=['#AB63FA'])
fig3.add_vline(x=0, line_dash='dash', line_color='red')
fig3.show()

**Feature Importance**

In [22]:
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_}).sort_values('Coefficient', ascending=False)

fig = px.bar(
    coef_df,
    x='Coefficient',
    y='Feature',
    orientation='h',
    color='Coefficient',
    color_continuous_scale='RdBu',
    height=600,
    width=800,
    title='Feature Coefficients (Importance)'
)
fig.show()

In [23]:
import joblib
joblib.dump(model,"Model.pkl")

['Model.pkl']